In [1]:
from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd
import torch


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'src' / 'M2F').exists() and (p / 'untracked').exists():
            return p
    raise RuntimeError('Could not find repo root containing src/M2F and untracked')


REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

from M2F.pyg_data_interfaces import DatasetInput, ProteinGraphInMemoryDataset
from M2F.embedding_utils import AAChainEmbedder
from M2F.cleaning_utils import clean_col
from M2F.feature_engineering_utils import encode_go, embed_AAsequences


# --------------------------- config ---------------------------
GO_DEPTH = 4
AA_MODEL_KEY = 'esm2_t6_8M_UR50D'
AA_BATCH_SIZE = 16
FORCE_RELOAD = True

subset_dir = REPO_ROOT / 'untracked' / 'test_data_subset'
out_root = REPO_ROOT / 'untracked' / 'prot1_real_uniprot_go_depth'
if FORCE_RELOAD:
    shutil.rmtree(out_root, ignore_errors=True)


# ---------------------- transform helpers ---------------------
go_label_map: dict[str, int] = {}
aa_device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
aa_encoder = AAChainEmbedder(model_key=AA_MODEL_KEY, device=aa_device)


def _as_go_multihot(idx_tuple, y_dim: int):
    if not isinstance(idx_tuple, tuple) or y_dim == 0:
        return np.nan
    vec = np.zeros(y_dim, dtype=np.float32)
    if idx_tuple:
        vec[np.asarray(idx_tuple, dtype=np.int64)] = 1.0
    return vec if vec.sum() > 0 else np.nan


def composed_pre_transform(node_df: pd.DataFrame) -> pd.DataFrame:
    df = node_df.copy()

    # M2F extraction/normalization.
    df = clean_col(df, 'Sequence', apply_norm=False, apply_strip_pubmed=False, inplace=True)
    df = clean_col(
        df,
        'Gene Ontology (molecular function)',
        apply_norm=False,
        apply_strip_pubmed=True,
        inplace=True,
    )

    # M2F GO depth encoding.
    df, labels = encode_go(
        df,
        col_name='Gene Ontology (molecular function)',
        depth=GO_DEPTH,
        inplace=True,
    )
    go_label_map.clear()
    go_label_map.update(labels)
    y_dim = len(go_label_map)

    # Fixed-width target vectors for tensor stacking.
    df.loc[:, 'Gene Ontology (molecular function)'] = df[
        'Gene Ontology (molecular function)'
    ].map(lambda t: _as_go_multihot(t, y_dim))

    # M2F AA sequence embedding.
    df = embed_AAsequences(df, embedder=aa_encoder, batch_size=AA_BATCH_SIZE, inplace=True)

    return df


def pre_filter_mask(df: pd.DataFrame):
    x_ok = df['Sequence'].map(
        lambda x: isinstance(x, np.ndarray) and x.size > 0 and np.isfinite(x).all()
    )
    y_ok = df['Gene Ontology (molecular function)'].map(
        lambda y: isinstance(y, np.ndarray) and y.size > 0 and np.isfinite(y).all() and (y.sum() > 0)
    )
    return x_ok & y_ok


# ------------------- dataset build + checks -------------------
dataset_input = DatasetInput(
    path_to_accession_ids_csv_file=subset_dir / 'uniref_index_count.csv',
    path_to_edge_csv_dir=subset_dir,
    X={'sequence': 'Sequence'},
    Y={'go_f': 'Gene Ontology (molecular function)'},
    edge_dst_column='j',
    edge_attr_columns=('v',),
    request_size=25,
    rps=1,
    max_retry=20,
)

ds = ProteinGraphInMemoryDataset(
    root=out_root,
    dataset_input=dataset_input,
    pre_transform=composed_pre_transform,
    pre_filter=pre_filter_mask,
    force_reload=FORCE_RELOAD,
)

data = ds[0]
print(f'nodes={data.num_nodes}, edges={data.num_edges}')
print(f'x={tuple(data.x.shape)}, y={tuple(data.y.shape)}, edge_attr={tuple(data.edge_attr.shape)}')
print(f'go_classes={len(go_label_map)} at depth={GO_DEPTH}')

assert data.num_nodes > 0
assert data.edge_index.shape[0] == 2
assert data.x.shape[0] == data.num_nodes
assert data.y.shape[0] == data.num_nodes
assert data.y.shape[1] == len(go_label_map)
print('Real UniProt + M2F transform/filter dataset build passed.')


/Users/yehormishchyriak/Desktop/BonhamLab/microbiome2function/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/yehormishchyriak/Desktop/BonhamLab/microbiome2function/src/M2F/dependencies/go-basic.obo: fmt(1.2) rel(2025-07-22) 43,230 Terms


Processing...


nodes=57, edges=124
x=(57, 320), y=(57, 53), edge_attr=(124, 1)
go_classes=53 at depth=4
Real UniProt + M2F transform/filter dataset build passed.


Done!
